In [4]:
import hashlib
import re
import unicodedata
from pathlib import Path

TARGET = "e23428da87648a4bfae34ea1c7fa97892fd998d5d308f56613cd8daafe7d1e40"

CHEESE_LIST_PATH = "cheese_list.txt"


def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def normalize_basic(s: str) -> list[str]:

    s = s.strip()

    # Quitar acentos (por seguridad)
    s_no_acc = "".join(
        c for c in unicodedata.normalize("NFKD", s)
        if not unicodedata.combining(c)
    )

    variants = [
        s,
        s.lower(),
        s_no_acc,
        s_no_acc.lower(),
        s.replace(" ", ""),
        s.lower().replace(" ", ""),
    ]

    # eliminar duplicados
    seen = set()
    out = []
    for v in variants:
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out


def try_crack(cheese: str):

    variants = normalize_basic(cheese)

    for v in variants:
        vb = v.encode()

        if sha256_hex(vb) == TARGET:
            return v, "NO_SALT", "sha256(cheese)"


        for salt in range(256):
            salt_str = str(salt)
            sb = salt_str.encode()

            combos = [
                (v + salt_str).encode(),          # cheese + "23"
                (salt_str + v).encode(),          # "23" + cheese
                f"{v}:{salt_str}".encode(),       # cheese:23
                f"{v}|{salt_str}".encode(),       # cheese|23
                f"{salt_str}:{v}".encode(),       # 23:cheese
            ]

            for data in combos:
                if sha256_hex(data) == TARGET:
                    return v, salt_str, "DECIMAL_SALT"

        for i in range(256):
            salt_hex = format(i, "02x")
            salt_ascii = salt_hex.encode()
            salt_byte = bytes.fromhex(salt_hex)

            combos = [
                vb + salt_ascii,
                salt_ascii + vb,
                vb + b":" + salt_ascii,
                vb + b"|" + salt_ascii,
                salt_ascii + b":" + vb,
                vb + salt_byte,
                salt_byte + vb,
            ]

            for data in combos:
                if sha256_hex(data) == TARGET:
                    return v, salt_hex, "HEX_OR_BYTE_SALT"

    return None


def main():
    path = Path(CHEESE_LIST_PATH)

    if not path.exists():
        print(f"[ERROR] No se encontró: {CHEESE_LIST_PATH}")
        print("Coloca cheese_list.txt en la misma carpeta del script.")
        return

    print("[*] Cargando lista de quesos...")
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        cheeses = [line.strip() for line in f if line.strip()]

    print(f"[*] Total de quesos: {len(cheeses)}")
    print("[*] Iniciando crack (optimizado para picoCTF)...\n")

    for idx, cheese in enumerate(cheeses, 1):
        result = try_crack(cheese)

        if result:
            variant, salt, mode = result
            print("¡HASH CRACKEADO!")
            print("=================================")
            print(f"Queso original  : {cheese}")
            print(f"Variante usada  : {variant}")
            print(f"Salt encontrado : {salt}")
            print(f"Modo            : {mode}")
            print("En el nc escribe EXACTAMENTE:")
            print(f"{variant}")
            return

        if idx % 50 == 0:
            print(f"[*] Progreso: {idx}/{len(cheeses)} quesos probados...")

    print("\n No se encontró coincidencia.")
    print("Posibles causas:")
    print("- El hash cambió al reconectar el nc")
    print("- La cheese_list.txt no es la correcta")
    print("- El reto usa otra forma de hash")


if __name__ == "__main__":
    main()

[*] Cargando lista de quesos...
[*] Total de quesos: 599
[*] Iniciando crack (optimizado para picoCTF)...

[*] Progreso: 50/599 quesos probados...
[*] Progreso: 100/599 quesos probados...
[*] Progreso: 150/599 quesos probados...
[*] Progreso: 200/599 quesos probados...
[*] Progreso: 250/599 quesos probados...
[*] Progreso: 300/599 quesos probados...
¡HASH CRACKEADO!
Queso original  : Hipi Iti
Variante usada  : hipi iti
Salt encontrado : fd
Modo            : HEX_OR_BYTE_SALT
En el nc escribe EXACTAMENTE:
hipi iti
